In [1]:
import pandas as pd
from bs4 import BeautifulSoup

df = pd.read_csv('/content/resume_dataset_1200.csv')

# Normalize all column names — strips spaces, fixes casing
df.columns = df.columns.str.strip()

print("Actual columns:", df.columns.tolist())

Actual columns: ['Name', 'Age', 'Gender', 'Education_Level', 'Field_of_Study', 'Degrees', 'Institute_Name', 'Graduation_Year', 'Experience_Years', 'Current_Job_Title', 'Previous_Job_Titles', 'Skills', 'Certifications', 'Target_Job_Description']


In [2]:
# Function to build plain text resume from structured columns
def structured_to_text(row):
    parts = []

    if pd.notna(row.get('Name')):            parts.append(f"Name: {row['Name']}")
    if pd.notna(row.get('Age')):             parts.append(f"Age: {row['Age']}")
    if pd.notna(row.get('Gender')):          parts.append(f"Gender: {row['Gender']}")
    if pd.notna(row.get('Education_Level')): parts.append(f"Education: {row['Education_Level']}")
    if pd.notna(row.get('Degrees')):         parts.append(f"Degree: {row['Degrees']}")
    if pd.notna(row.get('Field_of_Study')):  parts.append(f"Field: {row['Field_of_Study']}")
    if pd.notna(row.get('Institute_Name')):  parts.append(f"Institute: {row['Institute_Name']}")
    if pd.notna(row.get('Graduation_Year')): parts.append(f"Graduated: {int(row['Graduation_Year'])}")
    if pd.notna(row.get('Experience_Years')):parts.append(f"Experience: {row['Experience_Years']} years")
    if pd.notna(row.get('Current_Job_Title')):parts.append(f"Current Role: {row['Current_Job_Title']}")
    if pd.notna(row.get('Previous_Job_Titles')):parts.append(f"Previous Roles: {row['Previous_Job_Titles']}")
    if pd.notna(row.get('Skills')):          parts.append(f"Skills: {row['Skills']}")
    if pd.notna(row.get('Certifications')):  parts.append(f"Certifications: {row['Certifications']}")

    return "\n".join(parts)

# Apply to create resume_text column
df['resume_text'] = df.apply(structured_to_text, axis=1)

# Select relevant columns
extracted_df = df[['Name', 'resume_text', 'Current_Job_Title', 'Target_Job_Description']].copy()

# Display sample
print(extracted_df.head())

# Save to new CSV
extracted_df.to_csv('extracted_resumes.csv', index=False)

               Name                                        resume_text  \
0      Akash Pillai  Name: Akash Pillai\nAge: 30\nGender: Non-Binar...   
1  Charlotte Taylor  Name: Charlotte Taylor\nAge: 27\nGender: Non-B...   
2        James Zhou  Name: James Zhou\nAge: 45\nGender: Male\nEduca...   
3     Amelia Thomas  Name: Amelia Thomas\nAge: 28\nGender: Male\nEd...   
4       Amanda Jain  Name: Amanda Jain\nAge: 42\nGender: Non-Binary...   

        Current_Job_Title                             Target_Job_Description  
0                     NaN  Seeking a challenging role as a Software Devel...  
1  Cybersecurity Engineer  Targeting a Cybersecurity Engineer position to...  
2         Prompt Engineer  Targeting a Prompt Engineer position to utiliz...  
3             AI Engineer  Targeting a Data Scientist position where I ca...  
4  Cybersecurity Engineer  Looking for an opportunity as a Prompt Enginee...  


In [3]:
import pandas as pd
import re

# 1. Load data
df = pd.read_csv('extracted_resumes.csv')

# 2. Clean resume_text
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)

    # Remove control characters
    text = re.sub(r'[\r\t\x0b\x0c]', ' ', text)

    # Collapse multiple newlines and spaces
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'[ ]{2,}', ' ', text)

    # Normalize punctuation spacing
    text = re.sub(r'\s+([,.:;!?])', r'\1', text)
    text = re.sub(r'([,.:;!?])([^\s])', r'\1 \2', text)

    # Trim each line
    lines = [line.strip() for line in text.split('\n')]
    cleaned = '\n'.join([line for line in lines if line])

    # Smart casing: preserve acronyms, capitalize others
    def smart_case(s):
        return ' '.join([w if w.isupper() else w.capitalize() for w in s.split()])

    return smart_case(cleaned)

df['resume_text_clean'] = df['resume_text'].astype(str).apply(clean_text)

# 3. Derive weak labels from Current_Job_Title
def weak_label(row):
    title = str(row.get('Current_Job_Title', '')).lower()

    if any(k in title for k in ['director', 'manager', 'head', 'lead', 'vp', 'chief']):
        return 0.8
    if any(k in title for k in ['specialist', 'analyst', 'engineer', 'developer', 'architect']):
        return 0.7
    if any(k in title for k in ['administrator', 'coordinator', 'consultant', 'advisor']):
        return 0.6
    if any(k in title for k in ['intern', 'trainee', 'junior', 'assistant']):
        return 0.4
    return 0.5

if 'label' in df.columns:
    df['label'] = pd.to_numeric(df['label'], errors='coerce').clip(0.0, 1.0)
else:
    df['label'] = df.apply(weak_label, axis=1)

# 4. Preview
print(df[['Name', 'resume_text_clean', 'Current_Job_Title', 'label']].head())
print(f"\nLabel distribution:\n{df['label'].describe()}")
print(f"\nEmpty resume_text: {(df['resume_text_clean'].str.strip() == '').sum()}")

# 5. Save
df.to_csv('cleaned_resumes_with_labels.csv', index=False)
print("\nSaved: cleaned_resumes_with_labels.csv")

               Name                                  resume_text_clean  \
0      Akash Pillai  Name: Akash Pillai Age: 30 Gender: Non-binary ...   
1  Charlotte Taylor  Name: Charlotte Taylor Age: 27 Gender: Non-bin...   
2        James Zhou  Name: James Zhou Age: 45 Gender: Male Educatio...   
3     Amelia Thomas  Name: Amelia Thomas Age: 28 Gender: Male Educa...   
4       Amanda Jain  Name: Amanda Jain Age: 42 Gender: Non-binary E...   

        Current_Job_Title  label  
0                     NaN    0.5  
1  Cybersecurity Engineer    0.7  
2         Prompt Engineer    0.7  
3             AI Engineer    0.7  
4  Cybersecurity Engineer    0.7  

Label distribution:
count    1200.000000
mean        0.621583
std         0.118832
min         0.500000
25%         0.500000
50%         0.700000
75%         0.700000
max         0.800000
Name: label, dtype: float64

Empty resume_text: 0

Saved: cleaned_resumes_with_labels.csv


In [4]:
import pandas as pd

# 1. Load cleaned data
cleaned_df = pd.read_csv('cleaned_resumes_with_labels.csv')

# 2. Normalize job title as merge/lookup key
#    (replaces Category_norm — your dataset has no Category column)
cleaned_df['title_norm'] = (
    cleaned_df['Current_Job_Title']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

# 3. No external job CSV needed — Target_Job_Description already exists
#    Just rename for consistency with downstream cells
cleaned_df['job_description'] = cleaned_df['Target_Job_Description'].astype(str)

# 4. Generic fallback templates keyed by job title
default_templates = {
    'hr':                        "Responsible for recruitment, employee relations, and compliance.",
    'human resources':           "Manage talent acquisition, performance reviews, and HR policy.",
    'software engineer':         "Develop and maintain software applications using modern languages.",
    'data scientist':            "Analyze data, build predictive models, and derive business insights.",
    'data analyst':              "Collect, process, and analyze data to support business decisions.",
    'machine learning engineer': "Design and deploy ML pipelines and models into production.",
    'product manager':           "Define product roadmap, gather requirements, and coordinate teams.",
    'marketing specialist':      "Plan and execute digital marketing campaigns and analyze metrics.",
    'sales representative':      "Identify leads, engage prospects, and close sales deals.",
    'business analyst':          "Gather business requirements and perform data-driven analysis.",
    'devops engineer':           "Implement CI/CD pipelines and manage cloud infrastructure.",
    'ux designer':               "Conduct user research and create wireframes and prototypes.",
    'financial analyst':         "Perform financial modeling and reporting to support decision-making.",
    'project manager':           "Plan and execute projects on time and within budget.",
    'backend developer':         "Build and maintain server-side logic, APIs, and databases.",
    'frontend developer':        "Develop responsive UI components using modern JS frameworks.",
    'full stack developer':      "Design and implement both client-side and server-side features.",
    'cloud engineer':            "Architect and manage cloud solutions on AWS, Azure, or GCP.",
    'cybersecurity analyst':     "Monitor, detect, and respond to security threats.",
    'network engineer':          "Design, implement, and maintain enterprise network infrastructure.",
}

# 5. Fill missing job_description with template fallback
def fill_job_description(row):
    jd = str(row['job_description']).strip()

    if jd in ('', 'nan', 'None', 'NaN'):
        title = row['title_norm']

        # Exact match
        if title in default_templates:
            return default_templates[title]

        # Partial match
        for key, template in default_templates.items():
            if key in title or title in key:
                return template

        # Build from skills as last resort
        skills = str(row.get('Skills', '')).strip()
        if skills and skills not in ('nan', 'None'):
            return f"Professional with expertise in {skills}."

        return "Experienced professional seeking a challenging role."

    return jd

cleaned_df['job_description'] = cleaned_df.apply(fill_job_description, axis=1)

# 6. Drop helper column
final_df = cleaned_df.drop(columns=['title_norm'])

# 7. Save
final_df.to_csv('resumes_with_fallback_job_descriptions.csv', index=False)
print("Saved: resumes_with_fallback_job_descriptions.csv")

# 8. Preview
print(final_df[['Name', 'Current_Job_Title', 'job_description']].head(10).to_string())

# Sanity check
empty = (final_df['job_description'].str.strip().isin(['', 'nan', 'None'])).sum()
print(f"\nEmpty job_descriptions remaining: {empty}")
print(f"Total rows: {len(final_df)}")

Saved: resumes_with_fallback_job_descriptions.csv
               Name             Current_Job_Title                                                                                                                                                                                      job_description
0      Akash Pillai                           NaN                                        Seeking a challenging role as a Software Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.
1  Charlotte Taylor        Cybersecurity Engineer                                                      Targeting a Cybersecurity Engineer position to utilize my educational background and experience to drive results and achieve career objectives.
2        James Zhou               Prompt Engineer                                                             Targeting a Prompt Engineer position to utilize my educational background and experience to drive r

In [12]:
# !!! DO NOT EDIT THIS CELL. THIS IS A SUB-CELL. !!!
!pip install -q transformers peft PyPDF2 bitsandbytes accelerate jsonlines trl

## Data Split + JSONL Generation


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
import jsonlines
import json

# Load data
df = pd.read_csv('resumes_with_fallback_job_descriptions.csv')

# Validate required columns exist
required = ['resume_text_clean', 'job_description', 'label', 'Current_Job_Title']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Drop rows with empty resume or JD
df = df[df['resume_text_clean'].str.strip().notna()]
df = df[df['job_description'].str.strip().notna()]

# Normalize label to 0-100 scale for consistency
# label is 0.0-1.0 from weak_label, multiply to get 0-100
if df['label'].max() <= 1.0:
    df['label_score'] = (df['label'] * 100).round(1)
else:
    df['label_score'] = df['label'].clip(0, 100)

print(f"Total usable rows: {len(df)}")
print(f"Label score distribution:\n{df['label_score'].describe()}")

# Step 3: Split 80% train / 10% val / 10% test
# Stratify by Current_Job_Title instead of Category
# Bin titles to avoid single-sample strata errors
df['title_bin'] = df['Current_Job_Title'].astype(str).str.lower().str.strip()
title_counts = df['title_bin'].value_counts()
df['title_bin'] = df['title_bin'].apply(
    lambda t: t if title_counts[t] >= 5 else 'other'
)

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df['title_bin'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['title_bin'],
    random_state=42
)

# Step 4: Build fine-tuning records in Llama-3 chat format
def make_record(row):
    prompt = (
        f"Resume: {row['resume_text_clean']}\n\n"
        f"Job Description: {row['job_description']}"
    )

    target = json.dumps({
        "relevance_score": float(row['label_score'])
    })

    # Llama-3 instruct format
    formatted = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n"
        f"You are an ATS scoring assistant. "
        f"Return ONLY a JSON object with key 'relevance_score' between 0 and 100.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"{prompt}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
        f"{target}<|eot_id|>"
    )

    return {
        "text": formatted,           # for SFTTrainer
        "conversations": [           # for formatting_func
            {"role": "user",      "content": prompt},
            {"role": "assistant", "content": target}
        ]
    }

# Step 5: Generate and save JSONL files
splits = {
    'train':      train_df,
    'validation': val_df,
    'test':       test_df
}

for split_name, split_df in splits.items():
    records = [make_record(row) for _, row in split_df.iterrows()]
    with jsonlines.open(f"{split_name}_data.jsonl", mode='w') as writer:
        writer.write_all(records)

print(f"\nTrain: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
print("JSONL files saved: train_data.jsonl, validation_data.jsonl, test_data.jsonl")

Total usable rows: 1200
Label score distribution:
count    1200.000000
mean       62.158333
std        11.883151
min        50.000000
25%        50.000000
50%        70.000000
75%        70.000000
max        80.000000
Name: label_score, dtype: float64

Train: 960 | Validation: 120 | Test: 120
JSONL files saved: train_data.jsonl, validation_data.jsonl, test_data.jsonl


## LoRA Fine Tunning

In [14]:
import os
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from trl import SFTTrainer

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Check GPU capability — determines which dtype to use
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
use_bf16 = "A100" in gpu_name or "A10" in gpu_name or "H100" in gpu_name
print(f"GPU: {gpu_name}")
print(f"Using {'bf16' if use_bf16 else 'fp16 disabled (float32 LoRA)'}")

# FIX: compute dtype must match training precision
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,  # FIX: was hardcoded float16
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    use_cache=False,
)

model.gradient_checkpointing_enable()

for param in model.parameters():
    param.requires_grad = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_ds = load_dataset("json", data_files="train_data.jsonl")["train"]
val_ds   = load_dataset("json", data_files="validation_data.jsonl")["train"]
print(f"\nTrain: {len(train_ds)} | Val: {len(val_ds)}")

def formatting_func(example):
    messages = example['conversations']
    prompt = ""
    for msg in messages:
        role    = msg[0] if isinstance(msg, (list, tuple)) else msg.get('role')
        content = msg[1] if isinstance(msg, (list, tuple)) else msg.get('content')
        if role == "user":
            prompt += f"User: {content}\n"
        elif role == "assistant":
            prompt += f"Assistant: {content}\n"
    return prompt.strip()

training_args = TrainingArguments(
    output_dir="./tinyllama-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=False,              # FIX: disabled — causes BFloat16 scaler conflict
    bf16=use_bf16,           # FIX: only enable on A100+
    logging_steps=50,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    gradient_checkpointing=True,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    formatting_func=formatting_func,
    args=training_args,
)

print("\nStarting LoRA fine-tuning...")
trainer.train()

model.save_pretrained("./tinyllama-lora")
tokenizer.save_pretrained("./tinyllama-lora")
print("\nSaved to ./tinyllama-lora")

GPU: Tesla T4
Using fp16 disabled (float32 LoRA)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 6,307,840 || all params: 1,106,356,224 || trainable%: 0.5701


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



Train: 960 | Val: 120


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.



Starting LoRA fine-tuning...


Step,Training Loss,Validation Loss



Saved to ./tinyllama-lora


## Training Status Check

In [15]:
import os

output_dir = "./tinyllama-lora"  # updated from phi3mini

if os.path.exists(output_dir):
    print(f"Output directory '{output_dir}' exists.")

    # Check trainer state
    trainer_state_file = os.path.join(output_dir, "trainer_state.json")
    if os.path.exists(trainer_state_file):
        import json
        with open(trainer_state_file) as f:
            state = json.load(f)
        print(f"Trainer state found.")
        print(f"  Global step     : {state.get('global_step', 'N/A')}")
        print(f"  Best eval loss  : {state.get('best_metric', 'N/A')}")
        print(f"  Epochs completed: {state.get('epoch', 'N/A')}")
    else:
        print(f"trainer_state.json not found — trainer may not have completed initialization.")

    # List checkpoints
    checkpoints = [
        d for d in os.listdir(output_dir)
        if os.path.isdir(os.path.join(output_dir, d)) and d.startswith("checkpoint-")
    ]
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        print(f"\nFound checkpoints: {checkpoints}")
        print(f"Latest checkpoint: {checkpoints[-1]}")
    else:
        print("No checkpoints found in output directory.")

    # Check for final adapter files
    adapter_model = os.path.join(output_dir, "adapter_model.safetensors")
    adapter_config = os.path.join(output_dir, "adapter_config.json")

    if os.path.exists(adapter_model) and os.path.exists(adapter_config):
        size_mb = os.path.getsize(adapter_model) / (1024 * 1024)
        print(f"\nFinal adapter files found — fine-tuning completed.")
        print(f"  adapter_model.safetensors: {size_mb:.1f} MB")
    else:
        print("\nFinal adapter files NOT found — fine-tuning may not have completed.")
        print("  Try resuming from the latest checkpoint.")

else:
    print(f"Output directory '{output_dir}' does not exist.")
    print("Training has not started or the directory was not created.")

Output directory './tinyllama-lora' exists.
trainer_state.json not found — trainer may not have completed initialization.

Found checkpoints: ['checkpoint-90']
Latest checkpoint: checkpoint-90

Final adapter files found — fine-tuning completed.
  adapter_model.safetensors: 12.1 MB


## Evaluation On Test Set

In [16]:
import torch
import json
import re
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset
from sklearn.metrics import mean_squared_error, r2_score

# Model paths — updated for LLaMA-3
model_name  = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
output_dir  = "./tinyllama-lora"

# Load base model in 4-bit
print("Loading base model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Load LoRA adapter on top
print("Loading fine-tuned LoRA adapters...")
model = PeftModel.from_pretrained(base_model, output_dir)
model.eval()

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load test dataset
print("Loading test dataset...")
test_ds = load_dataset("json", data_files="test_data.jsonl")["train"]
print(f"Test samples: {len(test_ds)}")

# Prediction function — Llama-3 format
def predict(model, tokenizer, text: str, max_length: int = 1024) -> str:
    # Wrap in Llama-3 chat format
    formatted = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n"
        f"You are an ATS scoring assistant. "
        f"Return ONLY a JSON object with key 'relevance_score' between 0 and 100.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"{text}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            num_return_sequences=1,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

# Generate predictions and extract scores
predictions  = []
true_labels  = []
parse_failures = 0

print("\nGenerating predictions on test set...")
for i, example in enumerate(test_ds):
    prompt     = example['conversations'][0]['content']
    true_label = json.loads(example['conversations'][1]['content'])['relevance_score']

    prediction_text = predict(model, tokenizer, prompt)

    predicted_score = None
    try:
        # Try JSON parse first
        score_str = prediction_text.split('Assistant: ')[-1].strip()
        # Strip markdown fences if present
        score_str = re.sub(r'```json|```', '', score_str).strip()
        predicted_score = json.loads(score_str).get('relevance_score')

    except json.JSONDecodeError:
        # Try JSON substring extraction
        match = re.search(r'\{.*\}', prediction_text, re.DOTALL)
        if match:
            try:
                predicted_score = json.loads(match.group(0)).get('relevance_score')
            except:
                pass

        # Regex fallback
        if predicted_score is None:
            match = re.search(r'"relevance_score"\s*:\s*([\d.]+)', prediction_text)
            if match:
                try:
                    predicted_score = float(match.group(1))
                except ValueError:
                    print(f"  Could not convert score to float: {match.group(1)}")

        if predicted_score is None:
            print(f"  Could not find relevance_score in: {prediction_text[:100]}")

    except Exception as e:
        print(f"  Error on example {i}: {e}")
        parse_failures += 1

    if predicted_score is not None:
        predictions.append(float(predicted_score))
        true_labels.append(float(true_label))

    if (i + 1) % 25 == 0:
        print(f"  Processed {i+1}/{len(test_ds)} | Valid: {len(predictions)} | Failed: {parse_failures}")

# Calculate evaluation metrics
print("\n" + "=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)

if predictions and true_labels:
    min_len      = min(len(predictions), len(true_labels))
    predictions  = predictions[:min_len]
    true_labels  = true_labels[:min_len]

    mse  = mean_squared_error(true_labels, predictions)
    rmse = np.sqrt(mse)
    r2   = r2_score(true_labels, predictions)
    mae  = np.mean(np.abs(np.array(predictions) - np.array(true_labels)))

    # Agreement within 10 points (mirrors your notes metric)
    agreement = sum(abs(p - t) <= 10 for p, t in zip(predictions, true_labels))
    agreement_pct = agreement / len(predictions)

    print(f"  Samples evaluated       : {len(predictions)}/{len(test_ds)}")
    print(f"  Parse failure rate      : {parse_failures/len(test_ds):.1%}")
    print(f"  Mean Absolute Error     : {mae:.4f}")
    print(f"  Mean Squared Error      : {mse:.4f}")
    print(f"  Root MSE                : {rmse:.4f}")
    print(f"  R-squared               : {r2:.4f}")
    print(f"  Agreement (within ±10)  : {agreement_pct:.1%}")

    if agreement_pct >= 0.88:
        print("\n  Target of 88% agreement reached.")
    else:
        print(f"\n  Below 88% target — consider more training data or epochs.")
else:
    print("No valid predictions were generated for evaluation.")
    print("Check model output format and score extraction logic.")

Loading base model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading fine-tuned LoRA adapters...
Loading tokenizer...
Loading test dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Test samples: 120

Generating predictions on test set...
  Could not find relevance_score in: Job Description: Targeting a Marketing Manager position to utilize my educational background and exp
  Could not find relevance_score in: Job Description: Targeting a AI Ethics Officer position to utilize my educational background and exp
  Could not find relevance_score in: Job Description: Targeting a Cloud Engineer position to utilize my educational background and experi
  Could not find relevance_score in: Job Description: Targeting a Data Scientist position where I can utilize my expertise in Python, R, 
  Processed 25/120 | Valid: 21 | Failed: 0
  Could not find relevance_score in: Job Description: Targeting a AI Ethics Officer position to utilize my educational background and exp
  Could not find relevance_score in: Job Description: Targeting a Site Reliability Engineer position to utilize my educational background
  Processed 50/120 | Valid: 44 | Failed: 0
  Could not find relevance_sc

## Full Inference with enhanced output

In [17]:
import io
from google.colab import files


# PDF extraction utility
def extract_text_from_pdf(pdf_bytes: bytes) -> str:
    from PyPDF2 import PdfReader
    reader = PdfReader(io.BytesIO(pdf_bytes))
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text.strip()

# Prediction helper — Llama-3 format + enhanced output schema
def predict_full(model, tokenizer, prompt: str, max_length: int = 1024) -> str:
    formatted = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n"
        f"You are an ATS scoring assistant. Analyze the resume vs. job description "
        f"and output a JSON object with:\n"
        f"1. relevance_score: a number between 0 and 100\n"
        f"2. missing_keywords: a list of important keywords from the JD not in the resume\n"
        f"3. recommendations: a list of bullet-point suggestions for improving the resume\n"
        f"Example output:\n"
        f'{{"relevance_score": 75.4, '
        f'"missing_keywords": ["AWS", "API development"], '
        f'"recommendations": ["Add AWS cloud experience", "Include API development details"]}}'
        f"<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"{prompt}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

# Input method
print("Choose input method:")
print("  1 = Upload PDF resume")
print("  2 = Use row from your CSV dataset")
choice = input("Enter 1 or 2: ").strip()

if choice == "1":
    print("\n📄 Upload your resume PDF:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    resume_bytes = next(iter(uploaded.values()))
    resume_text  = extract_text_from_pdf(resume_bytes)
    print(f"Extracted {len(resume_text)} characters from PDF.")

elif choice == "2":
    import pandas as pd
    df  = pd.read_csv('resumes_with_fallback_job_descriptions.csv')
    idx = int(input(f"Enter row index (0 to {len(df)-1}): ").strip())
    row = df.iloc[idx]
    resume_text = str(row.get('resume_text_clean', row.get('resume_text', '')))
    print(f"\nCandidate : {row.get('Name', 'N/A')}")
    print(f"Role      : {row.get('Current_Job_Title', 'N/A')}")
else:
    raise ValueError("Invalid choice.")

# Job description input
print("\n📋 Paste job description (blank line to finish):")
lines = []
while True:
    line = input()
    if not line.strip():
        break
    lines.append(line)
job_description = "\n".join(lines)

if not job_description.strip() and choice == "2":
    job_description = str(row.get('job_description', ''))
    print("Using job description from dataset row.")

# Format prompt
prompt = (
    f"Job Description:\n{job_description}\n\n"
    f"Resume Text:\n{resume_text}"
)

# Generate prediction
print("\nGenerating ATS analysis...")
predicted_text = predict_full(model, tokenizer, prompt)

# Extract JSON fields
predicted_data = {}
try:
    clean = re.sub(r'```json|```', '', predicted_text).strip()
    predicted_data = json.loads(clean)
except json.JSONDecodeError:
    # Try JSON substring
    match = re.search(r'\{.*\}', predicted_text, re.DOTALL)
    if match:
        try:
            predicted_data = json.loads(match.group(0))
        except:
            pass

relevance_score   = predicted_data.get("relevance_score")
missing_keywords  = predicted_data.get("missing_keywords", [])
recommendations   = predicted_data.get("recommendations", [])

# Display results
print("\n" + "=" * 60)
print("ATS ANALYSIS RESULTS")
print("=" * 60)

if relevance_score is not None:
    score     = float(relevance_score)
    shortlist = score >= 60
    fit       = "Strong" if score >= 75 else "Moderate" if score >= 50 else "Weak"

    print(f"  Relevance Score : {score:.1f} / 100")
    print(f"  Shortlist       : {'YES' if shortlist else 'NO'}")
    print(f"  Fit Level       : {fit}")
else:
    print("  Could not extract relevance score.")

print("\nMissing Keywords:")
if missing_keywords:
    for kw in missing_keywords:
        print(f"  - {kw}")
else:
    print("  None detected.")

print("\nRecommendations to Improve Resume:")
if recommendations:
    for rec in recommendations:
        print(f"  - {rec}")
else:
    print("  None generated.")

print("=" * 60)

Choose input method:
  1 = Upload PDF resume
  2 = Use row from your CSV dataset
Enter 1 or 2: 1

📄 Upload your resume PDF:


Saving Raj_Abhishek_11_04_2025.pdf to Raj_Abhishek_11_04_2025.pdf
Extracted 4657 characters from PDF.

📋 Paste job description (blank line to finish):
data scientist
data scientist 
data scientist
data analyst
machine learning
ai
ai
data science
Data Scientist



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Generating ATS analysis...

ATS ANALYSIS RESULTS
  Relevance Score : 50.0 / 100
  Shortlist       : NO
  Fit Level       : Moderate

Missing Keywords:
  None detected.

Recommendations to Improve Resume:
  None generated.
